# Day 13, Step 2 — Harness Engineering: a permission guardrail and an observability trace

**Problem:** a harness is "everything around the model (or, here, around one requested action) that keeps it from running wild": what it is explicitly allowed to touch, and a record of what it actually did. `example.py` shows the harness deciding *when to stop*; this notebook shows the harness deciding *what is allowed to run at all*, plus keeping a trace so a run can be reconstructed after the fact instead of guessed at.

This notebook stays fully local, like `example.py` — no `ollama` call. `ALLOWED_ACTIONS` is a plain Python set: a fixed allowlist the graph checks against, not a LangChain tool.

**Graph:** `START -> check_permission -> (allowed) execute_action / (blocked) reject_action -> END`

## Setup — declaring the state

`example.py` uses `StateGraph(dict)` — an unlabeled dictionary, so LangGraph treats the whole state as one blob and a node's return **replaces it entirely**. This notebook instead declares a `State` class listing every field up front, the same pattern Day 12's notebooks use. With a named schema, LangGraph gives each field its own channel and merges a node's return **key by key** — so a node only returns the field(s) it actually changed, and everything else (like `requested_action`) carries forward automatically. `ALLOWED_ACTIONS` is the harness's system of record for what this graph may do.

In [1]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

ALLOWED_ACTIONS = {"lookup_order", "check_shipping_status"}


class State(TypedDict):
    requested_action: str
    trace: list
    allowed: bool
    output: str


## `check_permission` — the guardrail

Checks the requested action against the fixed allowlist before anything runs, and appends the decision to `trace` — the first entry in this run's observability record. It returns only `trace` and `allowed`; `requested_action` is untouched, so it simply carries forward in its own channel for later nodes to read.

In [2]:
def check_permission(state: State):
    allowed = state["requested_action"] in ALLOWED_ACTIONS
    if allowed:
        note = f"check_permission: '{state['requested_action']}' is allowed"
    else:
        note = f"check_permission: '{state['requested_action']}' is blocked"
    trace = state["trace"] + [note]
    return {"trace": trace, "allowed": allowed}


## `route_after_permission`

A plain rule, not a model decision — `example.py`'s conditional edges are deterministic too, and this notebook keeps that same style. The guardrail's verdict, not the requested action itself, decides which branch runs next.

In [3]:
def route_after_permission(state: State):
    return "execute" if state["allowed"] else "reject"


## `execute_action` and `reject_action` — the two possible outcomes

Both branches append one more line to `trace` and set `output`; `requested_action` is still just carried forward, unread by anything past this point. Neither branch can run the other's action — the guardrail already decided that upstream.

In [4]:
def execute_action(state: State):
    trace = state["trace"] + [f"execute_action: ran '{state['requested_action']}'"]
    return {"trace": trace, "output": f"'{state['requested_action']}' completed."}


def reject_action(state: State):
    trace = state["trace"] + [
        f"reject_action: refused '{state['requested_action']}' (not on the allowed list)"
    ]
    return {
        "trace": trace,
        "output": f"'{state['requested_action']}' was blocked by the harness.",
    }


## Build the graph

In [5]:
graph_builder = StateGraph(State)
graph_builder.add_node("check_permission", check_permission)
graph_builder.add_node("execute_action", execute_action)
graph_builder.add_node("reject_action", reject_action)
graph_builder.add_edge(START, "check_permission")
graph_builder.add_conditional_edges(
    "check_permission",
    route_after_permission,
    {"execute": "execute_action", "reject": "reject_action"},
)
graph_builder.add_edge("execute_action", END)
graph_builder.add_edge("reject_action", END)
graph = graph_builder.compile()


## Run it twice

`lookup_order` is on the allowlist; `issue_refund` is not. Watch the printed `trace` — that is the observability record this notebook builds up, and it is what lets you reconstruct exactly what the harness did and why, after the run is over, instead of guessing.

In [8]:
def run(requested_action):
    result = graph.invoke({"requested_action": requested_action, "trace": []})
    print(f"Requested action: {requested_action}")
    for line in result["trace"]:
        print(f"  - {line}")
    print(f"Output: {result['output']}\n")


run("lookup_order")
run("issue_refund")


Requested action: lookup_order
  - check_permission: 'lookup_order' is allowed
  - execute_action: ran 'lookup_order'
Output: 'lookup_order' completed.

Requested action: issue_refund
  - check_permission: 'issue_refund' is blocked
  - reject_action: refused 'issue_refund' (not on the allowed list)
Output: 'issue_refund' was blocked by the harness.

